# Imports

In [3]:
import xarray as xr
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client

# PBS

In [4]:
cluster = PBSCluster(
    cores=4, # The number of cores you want
    memory='64GB', # Amount of memory (resource_spec is the one)
    processes=1, # How many processes
    queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    local_directory='$TMPDIR', # Use your local directory
    account='P93300313', # Input your project ID here
    walltime='02:00:00', # Amount of wall time
)
cluster.scale(jobs=5)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.189:36347,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [5]:
# chunks='auto' appproaches this value
dask.config.set({'array.chunk-size': '8 GiB'})

# Load

In [6]:
Uds = xr.open_zarr('/glade/work/acruz/CESM/CESM_tropic_U', decode_times=True)
Uds = Uds.chunk(chunks='auto')
Uds

<xarray.Dataset> Size: 57GB
Dimensions:  (member: 100, time: 1260, lev: 32, lat: 22, lon: 161)
Coordinates:
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
  * time     (time) object 10kB 1910-02-01 00:00:00 ... 2015-01-01 00:00:00
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
Data variables:
    U        (member, time, lev, lat, lon) float32 57GB dask.array<chunksize=(15, 1260, 32, 22, 161), meta=np.ndarray>

In [7]:
Wds = xr.open_zarr('/glade/work/acruz/CESM/CESM_tropic_W')
Wds = Wds.chunk(chunks='auto')
Wds

<xarray.Dataset> Size: 57GB
Dimensions:  (member: 100, time: 1260, lev: 32, lat: 22, lon: 161)
Coordinates:
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
  * time     (time) object 10kB 1910-02-01 00:00:00 ... 2015-01-01 00:00:00
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
Data variables:
    OMEGA    (member, time, lev, lat, lon) float32 57GB dask.array<chunksize=(15, 1260, 32, 22, 161), meta=np.ndarray>

# Calculate

In [8]:
Uclim = Uds.groupby('time.month').mean(dim='time')
Uclim = Uclim.chunk(chunks='auto')
Uclim

<xarray.Dataset> Size: 544MB
Dimensions:  (month: 12, member: 100, lev: 32, lat: 22, lon: 161)
Coordinates:
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
Data variables:
    U        (month, member, lev, lat, lon) float32 544MB dask.array<chunksize=(12, 100, 32, 22, 161), meta=np.ndarray>

In [9]:
Uanom = Uds.groupby('time.month') - Uclim
# faster when saving 10+GiB to use zarr and rechunk
Uanom = Uanom.chunk(chunks='auto')
Uanom

<xarray.Dataset> Size: 57GB
Dimensions:  (lat: 22, lev: 32, lon: 161, member: 100, time: 1260)
Coordinates:
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
  * time     (time) object 10kB 1910-02-01 00:00:00 ... 2015-01-01 00:00:00
    month    (time) int64 10kB dask.array<chunksize=(1260,), meta=np.ndarray>
Data variables:
    U        (member, time, lev, lat, lon) float32 57GB dask.array<chunksize=(100, 180, 32, 22, 161), meta=np.ndarray>

In [10]:
Uanom.to_zarr('/glade/work/acruz/CESM/anomalies/CESM_tropic_U', mode='w', consolidated=True)

/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [11]:
Wclim = Wds.groupby('time.month').mean(dim='time')
Wclim = Wclim.chunk(chunks='auto')
Wclim

<xarray.Dataset> Size: 544MB
Dimensions:  (month: 12, member: 100, lev: 32, lat: 22, lon: 161)
Coordinates:
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
Data variables:
    OMEGA    (month, member, lev, lat, lon) float32 544MB dask.array<chunksize=(12, 100, 32, 22, 161), meta=np.ndarray>

In [12]:
Wanom = Wds.groupby('time.month') - Wclim
Wanom = Wanom.chunk(chunks='auto')
Wanom

<xarray.Dataset> Size: 57GB
Dimensions:  (lat: 22, lev: 32, lon: 161, member: 100, time: 1260)
Coordinates:
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
  * time     (time) object 10kB 1910-02-01 00:00:00 ... 2015-01-01 00:00:00
    month    (time) int64 10kB dask.array<chunksize=(1260,), meta=np.ndarray>
Data variables:
    OMEGA    (member, time, lev, lat, lon) float32 57GB dask.array<chunksize=(100, 180, 32, 22, 161), meta=np.ndarray>

In [13]:
Wanom.to_zarr('/glade/work/acruz/CESM/anomalies/CESM_tropic_W', mode='w', consolidated=True)

/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [15]:
# meridional mean at the end in case it fails i dont loose anoms
mean_Uanom = Uanom.mean(dim='lat')
mean_Uanom.to_netcdf('/glade/work/acruz/CESM/Uanom_tropic_slice.nc')

ERROR! Session/line number was not unique in database. History logging moved to new session 521


In [18]:
mean_Wanom = Wanom.mean(dim='lat')
mean_Wanom.to_netcdf('/glade/work/acruz/CESM/Wanom_tropic_slice.nc')

In [19]:
client.shutdown()